# Data setup and EDA

Runs the manuscript's data steps on Colab: download the three datasets, check the folder labels,
build the 70/15/15 splits and run the EDA (section 3.5).

Before starting: **Runtime > Change runtime type > GPU** (needed later for training, not for this notebook),
and have your Kaggle API token (`kaggle.json`, from Kaggle > Settings > Create New Token) ready.

## 1. Get the code

In [2]:
BRANCH = "main"
!git clone -b $BRANCH https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd repo
!pip install -q -e ".[train,dev]" kaggle

Cloning into 'repo'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 81 (delta 14), reused 71 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 37.64 KiB | 12.55 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for attribution (pyproject.toml) ... done


## 2. Optional: keep the datasets on Google Drive
Skip this on the free 15 GB Drive plan: the three datasets are well over 15 GB. Without it they go on
Colab's own disk (~70 GB free) and are re-downloaded each session, which takes a few minutes.

In [3]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/thesis-data/raw
!rm -rf data/raw && ln -s /content/drive/MyDrive/thesis-data/raw data/raw

Mounted at /content/drive


## 3. Kaggle token
On Kaggle: Settings > API Tokens > **Legacy API Credentials**. That downloads `kaggle.json`.

In [4]:
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


## 4. Download
Takes 20-40 minutes, mostly unzipping. Don't stop it partway: a stopped dataset is re-downloaded on the next run.

In [2]:
!python scripts/download_datasets.py

kannan_ai_human_art: downloading kausthubkannan/ai-and-human-art-classification -> data/raw/kannan
Dataset URL: https://www.kaggle.com/datasets/kausthubkannan/ai-and-human-art-classification
License(s): DbCL-1.0
100% 2.21G/2.21G [02:11<00:00, 18.0MB/s]

superpotato9_ai_recognition: downloading superpotato9/dalle-recognition-dataset -> data/raw/superpotato9
Dataset URL: https://www.kaggle.com/datasets/superpotato9/dalle-recognition-dataset
License(s): MIT
100% 13.4G/13.4G [13:26<00:00, 17.9MB/s]

ai_artbench: downloading ravidussilva/real-ai-art -> data/raw/ai_artbench
Dataset URL: https://www.kaggle.com/datasets/ravidussilva/real-ai-art
License(s): other
100% 9.95G/9.95G [10:13<00:00, 17.4MB/s]



## 5. Check the folder labels
Every folder should show `human` or `ai`. If any show `UNMATCHED` or the wrong label,
fix the `classes` rules for that dataset in `configs/default.yaml` and re-run this cell.

In [ ]:
!python scripts/inspect_datasets.py


== kannan_ai_human_art (data/raw/kannan)
     43442  UNMATCHED  ai_art_classification/test
     10330  ai         ai_art_classification/train/AI_GENERATED
      8288  human      ai_art_classification/train/NON_AI_GENERATED

== superpotato9_ai_recognition (data/raw/superpotato9)
     17855  ai         fakeV2/fake-v2
      3780  human      real

== ai_artbench (data/raw/ai_artbench)


## 6. Build the splits

In [ ]:
!python scripts/prepare_data.py

## 6b. Harmonize size and format
Resizes every split image to 256 px (center crop) and re-saves it as JPEG quality 95 under `data/harmonized/`.
Without this, the CNN can tell classes apart by file format (Superpotato9: human mostly JPEG, AI mostly PNG)
or resolution (AI-ArtBench: human 256 px, AI 256/768 px) instead of real AI artifacts.

In [ ]:
!python scripts/harmonize.py

## 7. EDA
Runs on the harmonized images and writes `outputs/eda/report.json` (class balance, image sizes, intensity/contrast,
normalization mean/std, leakage check) and `outputs/eda/per_image.csv`. Copy `train_normalization` into
`preprocess.mean` / `preprocess.std` in the config. Add `--raw` to analyze the original files instead
(written to `outputs/eda_raw/`), e.g. to show the before/after in the thesis.

In [ ]:
!python scripts/eda.py --max-per-split 2000

In [ ]:
import json, pandas as pd
report = json.load(open("outputs/eda/report.json"))
print(json.dumps(report["class_balance"], indent=1))
print("train normalization:", report["train_normalization"])
print("cross-split duplicates:", report["cross_split_duplicates"])
df = pd.read_csv("outputs/eda/per_image.csv")
df.groupby(["source", "label"])[["width", "height", "intensity", "contrast"]].describe().T

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, name in [(0, "human"), (1, "ai")]:
    d = df[df.label == label]
    axes[0].hist(d.intensity, bins=40, alpha=0.6, label=name)
    axes[1].hist(d.contrast, bins=40, alpha=0.6, label=name)
axes[0].set_title("Mean intensity"); axes[1].set_title("Contrast (luminance std)")
axes[0].legend(); plt.tight_layout()